# Study 875 — Idiosyncratic-Vol Change — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the level-vs-change additivity regression, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4104, 'spread_bps': 0.87, 't_nw': 0.86, 't_1s': 0.85, 'lo_bps': 8.43, 'hi_bps': 7.56, 'welch_t': 0.33, 'gross_sharpe': 0.21, 'placebo_obs': 0.87, 'placebo_mean': -0.014, 'placebo_sd': 0.886, 'placebo_p': 0.165, 'placebo_sigma': 0.99, 'placebo_draws': 1000, 'level_bps': -3.85, 'level_t': -2.85, 'add_corr': 0.216, 'add_beta': 0.158, 'alpha_bps': 1.47, 'alpha_t': 1.52, 'era_early_bps': 2.86, 'era_early_t': 2.31, 'era_early_n': 1970, 'era_late_bps': -0.98, 'era_late_t': -0.63, 'era_late_n': 2134, 'timer_1_gross': 0.87, 'timer_1_cost': 2.14, 'timer_1_net': -1.27, 'timer_1_t': -1.25, 'timer_5_gross': 0.87, 'timer_5_cost': 10.14, 'timer_5_net': -9.27, 'timer_5_t': -9.12, 'null_mean_t': -0.16, 'null_sd_t': 0.95, 'null_fire': 1, 'planted_t': 8.43, 'planted_welch': 4.15}

## The headline — long-falling-idio-vol / short-rising-idio-vol spread

Daily equal-weight bottom-30% minus top-30% delta-IVOL spread. The market factor is the equal-weight cross-sectional mean return; idio vol is the CAPM residual vol via `var(r) − cov(r,mkt)²/var(mkt)`.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : falling {R['lo_bps']:+.2f} vs rising {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : +0.87 bps/day  NW(10) t = +0.86  one-sample t = +0.85
books         : falling +8.43 vs rising +7.56 bps (Welch t = +0.33)
gross Sharpe  : 0.21 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.5f}")
print(f"observed sits {R['placebo_sigma']:+.2f} sigma from the placebo mean")

observed +0.87 bps vs placebo mean -0.014 (sd 0.886) -> p = 0.16500
observed sits +0.99 sigma from the placebo mean


## Additivity — is the CHANGE anything beyond the idio-vol LEVEL (501)?

Build the idio-vol *level* sort on the same tape, regress the change spread on it, and read the residual (Newey-West) *t*.

In [4]:
print(f"idio-vol LEVEL spread : {R['level_bps']:+.2f} bps/day (NW t = {R['level_t']:+.2f})")
print(f"corr(change, level)   : {R['add_corr']:+.3f}   beta = {R['add_beta']:+.3f}")
print(f"change alpha vs level : {R['alpha_bps']:+.2f} bps/day (NW t = {R['alpha_t']:+.2f})")

idio-vol LEVEL spread : -3.85 bps/day (NW t = -2.85)
corr(change, level)   : +0.216   beta = +0.158
change alpha vs level : +1.47 bps/day (NW t = +1.52)


## Robustness — two eras (split 2018-01-01)

The decisive cut: a full-sample *t* carried by one era and reversing in the other is not a signal.

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1970): +2.86 bps  NW t = +2.31
2018-2026 (n=2134): -0.98 bps  NW t = -0.63


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [6]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross +0.87 -> net -1.27 bps/day (cost 2.14/day, t=-1.25)
5 bps one-way: gross +0.87 -> net -9.27 bps/day (cost 10.14/day, t=-9.12)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT (reliably) fire on the null and must recover a planted relation.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from ivol_change import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=875+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.002, seed=875, n_assets=40, n_days=1500))
print(f"planted (edge=0.002): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.14 (sd 0.92), |t|>=2 in 0/8


planted (edge=0.002): NW t = +8.43, Welch t = +4.15


## Verdict

- **Signal — None.** The idio-vol *change* — distinct from the idio-vol level puzzle (corr with the level spread just +0.216) — carries **no** reliable cross-sectional signal on 50 liquid US mega-caps: the long-falling / short-rising spread is **+0.87 bps/day** (NW *t* = **+0.86**), the *claimed* sign but statistically zero (~1.0σ into the placebo), and — decisively — **not robust across eras** (*t* = +2.31 then -0.63, sign-flipped). It adds only +1.47 bps/day (*t* = +1.52) on top of the (itself inverted) level effect. The 20-seed synthetic control recovers a *planted* relation cleanly (*t* = +8.43, fires on 1/20 nulls — the nominal 5%), so the flat result is real, not machinery.
- **Tradability — Mirage.** The +0.87 bps/day gross edge is smaller than the 2.14 bps/day round-trip friction at 1 bp one-way, net **-1.27 bps/day** (*t* = -1.25); at 5 bps **-9.27 bps/day**. Nothing to trade.